# Promoter DP-SAT ε = 4, 2, 8

**Curated extraction from the original 124-cell notebook.** Outputs removed. This notebook has **not been executed end-to-end**; original cells are preserved where noted. Training is costly and changes checkpoints. DP-SAT source uses previous-gradient perturbation, Poisson sampling, and a privacy accountant; implementation guarantees remain unverified.

## Environment and data
Run in **Google Colab with GPU**. Restart runtime after installing if incompatible libraries are already imported. Check paths before training; original source saves to Google Drive.

In [ ]:
%pip install -q "transformers==4.49.0" "tokenizers==0.21.4" "huggingface-hub==0.36.2" "datasets==3.6.0" "opacus==1.6.0" "pandas==2.2.3" "accelerate" "scikit-learn" "matplotlib" "tqdm" "jedi>=0.16"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/DNABERT_DP_Project/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
import random,numpy as np,torch
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
MODEL_ID = 'zhihan1996/DNABERT-2-117M'
class DNABERTBiTFiTClassifier(nn.Module):
    def __init__(self, encoder, num_labels=2):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters(): p.requires_grad = False
        for name,p in self.encoder.named_parameters():
            if name.endswith('.bias'): p.requires_grad = True
        self.classifier = nn.Linear(self.encoder.config.hidden_size,num_labels)
    def forward(self,input_ids,attention_mask=None):
        outputs=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        hidden=outputs[0]
        if attention_mask is not None:
            mask=attention_mask.unsqueeze(-1).to(hidden.dtype)
            pooled=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
        else: pooled=hidden.mean(1)
        return self.classifier(pooled)
def fresh_encoder():
    encoder=AutoModel.from_pretrained(MODEL_ID,trust_remote_code=True,add_pooling_layer=False)
    for module in encoder.modules():
        if module.__class__.__name__=='BertUnpadSelfAttention':
            module.p_dropout=0.1
            if hasattr(module,'dropout'): module.dropout.p=0.1
    return encoder

**Original source cell 68** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 50 - RESTORE ORIGINAL PROMOTER DATASET
# Recreate the original 42,620 / 4,736 train/validation split
# ============================================================

import os
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

MODEL_ID = "zhihan1996/DNABERT-2-117M"
SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 8

print("=" * 68)
print("CELL 50 - RESTORE PROMOTER DATASET")
print("=" * 68)

# ------------------------------------------------------------
# 1. Load original GUE promoter dataset
# ------------------------------------------------------------

print("\n1. LOADING GUE PROMOTER DATASET")
print("-" * 68)

promoter = load_dataset(
    "leannmlindsey/GUE",
    "prom_core_all"
)

print(promoter)

original_train = promoter["train"]
original_test = promoter["test"]

print("\nOriginal training examples:", len(original_train))
print("Original test examples:", len(original_test))

assert len(original_train) == 47356
assert len(original_test) == 5920

# ------------------------------------------------------------
# 2. Recreate stratified train/validation split
# ------------------------------------------------------------

print("\n2. RECREATING TRAIN/VALIDATION SPLIT")
print("-" * 68)

indices = np.arange(len(original_train))
labels = np.asarray(original_train["label"])

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

promoter_train = original_train.select(train_idx.tolist())
promoter_val = original_train.select(val_idx.tolist())
promoter_test = original_test

print("Training:", len(promoter_train))
print("Validation:", len(promoter_val))
print("Test:", len(promoter_test))

assert len(promoter_train) == 42620
assert len(promoter_val) == 4736
assert len(promoter_test) == 5920

# ------------------------------------------------------------
# 3. Load tokenizer
# ------------------------------------------------------------

print("\n3. TOKENIZER")
print("-" * 68)

promoter_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=True
)

print("Tokenizer:", promoter_tokenizer.__class__.__name__)

# ------------------------------------------------------------
# 4. Tokenize all three splits
# ------------------------------------------------------------

def tokenize_promoter(batch):

    encoded = promoter_tokenizer(
        batch["sequence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    encoded["labels"] = batch["label"]

    return encoded


promoter_train_dataset = promoter_train.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_train.column_names,
    desc="Tokenizing promoter training"
)

promoter_val_dataset = promoter_val.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_val.column_names,
    desc="Tokenizing promoter validation"
)

promoter_test_dataset = promoter_test.map(
    tokenize_promoter,
    batched=True,
    remove_columns=promoter_test.column_names,
    desc="Tokenizing promoter test"
)

# ------------------------------------------------------------
# 5. Build evaluation loaders
# ------------------------------------------------------------

promoter_collator = DataCollatorWithPadding(
    tokenizer=promoter_tokenizer,
    padding=True,
    return_tensors="pt"
)

# IMPORTANT:
# This ordinary DataLoader is for inspection/evaluation.
# It is NOT the Poisson sampler for private training.

promoter_val_loader = DataLoader(
    promoter_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

promoter_test_loader = DataLoader(
    promoter_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=promoter_collator
)

# ------------------------------------------------------------
# 6. Final verification
# ------------------------------------------------------------

print("\n4. RESTORED DATASET VERIFICATION")
print("-" * 68)

print("Training examples:", len(promoter_train_dataset))
print("Validation examples:", len(promoter_val_dataset))
print("Test examples:", len(promoter_test_dataset))

print(
    "Training label distribution:",
    np.bincount(
        np.asarray(promoter_train_dataset["labels"])
    ).tolist()
)

print(
    "Validation label distribution:",
    np.bincount(
        np.asarray(promoter_val_dataset["labels"])
    ).tolist()
)

sample = promoter_train_dataset[0]

print("Tokenized fields:", list(sample.keys()))
print("First sequence token count:", len(sample["input_ids"]))

assert len(promoter_train_dataset) == 42620
assert len(promoter_val_dataset) == 4736
assert len(promoter_test_dataset) == 5920

print("\n" + "=" * 68)
print("CELL 50 COMPLETE - PROMOTER DATASET RESTORED")
print("=" * 68)

In [ ]:
# Compatibility aliases used by the original DP-SGD training cells.
train_dataset = promoter_train_dataset
val_dataset = promoter_val_dataset
test_dataset = promoter_test_dataset
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,collate_fn=promoter_collator)
eval_loader = promoter_val_loader
test_loader = promoter_test_loader
classifier_model = DNABERTBiTFiTClassifier(fresh_encoder(),num_labels=2).to(device)
print('Training:',len(train_dataset),'Validation:',len(val_dataset),'Test:',len(test_dataset))

**Static audit correction:** The compatibility DataLoader now uses `promoter_collator` defined by the dataset setup cell, replacing an undefined `collator` name. This notebook has not been run end-to-end.


## Original experiment cells
**CAUTION:** The following cells were extracted from different Colab sessions. Review paths, variable dependencies, privacy assumptions, and checkpoint names before running. Run one training budget at a time.

**Original source cell 63** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 45 - DP-SAT UPDATE RULE VALIDATION
# Park et al. (2023)
# No dataset access, model training, or checkpoint changes
# ============================================================

import torch
import json
import os

print("=" * 65)
print("CELL 45 - DP-SAT UPDATE RULE VALIDATION")
print("=" * 65)

torch.manual_seed(42)

# ------------------------------------------------------------
# 1. Experiment settings
# ------------------------------------------------------------

RHO = 0.01
TAU = 1e-12

# Synthetic trainable parameters
parameters = [
    torch.nn.Parameter(torch.randn(8)),
    torch.nn.Parameter(torch.randn(4, 3))
]

# Simulate a previous PRIVATIZED gradient.
# In real DP-SAT, this comes from the previous DP training step.
previous_private_gradients = [
    torch.randn_like(p) for p in parameters
]

# ------------------------------------------------------------
# 2. Implement DP-SAT perturbation
# ------------------------------------------------------------

def apply_dpsat_perturbation(
    parameters,
    previous_private_gradients,
    rho,
    tau=1e-12
):
    """
    Temporarily perturb parameters in the direction
    of the previous step's privatized gradient.

    Does not access training data or calculate gradients.
    """

    assert len(parameters) == len(previous_private_gradients)

    norm = torch.sqrt(sum(
        torch.sum(g.detach().float() ** 2)
        for g in previous_private_gradients
    ))

    scale = rho / (norm + tau)

    perturbations = []

    with torch.no_grad():
        for p, g in zip(parameters, previous_private_gradients):
            delta = scale * g
            p.add_(delta)
            perturbations.append(delta.clone())

    return perturbations


def remove_dpsat_perturbation(parameters, perturbations):
    """Restore parameters before applying the optimizer update."""

    with torch.no_grad():
        for p, delta in zip(parameters, perturbations):
            p.sub_(delta)


# ------------------------------------------------------------
# 3. Test perturbation and restoration
# ------------------------------------------------------------

original = [p.detach().clone() for p in parameters]

perturbations = apply_dpsat_perturbation(
    parameters,
    previous_private_gradients,
    rho=RHO,
    tau=TAU
)

perturbation_norm = torch.sqrt(sum(
    torch.sum(delta.float() ** 2)
    for delta in perturbations
)).item()

print("\n1. PERTURBATION TEST")
print("-" * 65)
print("Target rho:", RHO)
print("Actual perturbation norm:", perturbation_norm)

assert abs(perturbation_norm - RHO) < 1e-6

remove_dpsat_perturbation(parameters, perturbations)

restored = all(
    torch.allclose(p, original_p, atol=1e-6, rtol=1e-6)
    for p, original_p in zip(parameters, original)
)

print("\n2. RESTORATION TEST")
print("-" * 65)
print("Parameters restored:", restored)

assert restored, "Parameter restoration failed."

# ------------------------------------------------------------
# 4. Check DP-SGD equivalence when rho = 0
# ------------------------------------------------------------

zero_perturbations = apply_dpsat_perturbation(
    parameters,
    previous_private_gradients,
    rho=0.0,
    tau=TAU
)

zero_perturbation = all(
    torch.count_nonzero(delta).item() == 0
    for delta in zero_perturbations
)

remove_dpsat_perturbation(parameters, zero_perturbations)

print("\n3. DP-SGD EQUIVALENCE TEST")
print("-" * 65)
print("rho = 0 produces no perturbation:", zero_perturbation)

assert zero_perturbation

# ------------------------------------------------------------
# 5. Save validation results
# ------------------------------------------------------------

results = {
    "method": "DP-SAT",
    "paper": "Park et al. (2023)",
    "rho": RHO,
    "tau": TAU,
    "perturbation_norm": perturbation_norm,
    "parameters_restored": restored,
    "rho_zero_no_perturbation": zero_perturbation,
    "test_type": "Synthetic update-rule unit test",
    "privacy_accounting_verified": False,
    "real_model_training_tested": False
}

results_dir = (
    "/content/drive/MyDrive/DNABERT_DP_Project/results"
)

if os.path.isdir(results_dir):
    output_path = os.path.join(
        results_dir,
        "dnabert_dpsat_update_validation.json"
    )

    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)

    print("\nSaved:", output_path)
else:
    print("\nDrive not mounted; results not saved.")

print("\n" + "=" * 65)
print("CELL 45 COMPLETE - ALL UNIT TESTS PASSED")
print("=" * 65)

**Original source cell 66** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 48 - INDEPENDENT PRIVACY ACCOUNTING CHECK
# No model loading or training
# ============================================================

import sys
import subprocess
import importlib.util
import json
import os
import math
import pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/DNABERT_DP_Project/results"

N = 42620
BATCH_SIZE = 8
STEPS = 15984
DELTA = 1e-5

SAMPLE_RATE = BATCH_SIZE / N

print("=" * 68)
print("CELL 48 - INDEPENDENT PRIVACY ACCOUNTING")
print("=" * 68)

# Install Opacus without upgrading PyTorch or Transformers.
if importlib.util.find_spec("opacus") is None:
    print("\nInstalling Opacus 1.6.0 without dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "--no-deps", "opacus==1.6.0"
    ])

from opacus.accountants import RDPAccountant, PRVAccountant
import opacus

print("Opacus version:", opacus.__version__)
print("Sampling rate:", SAMPLE_RATE)
print("Steps:", STEPS)
print("Delta:", DELTA)

records = []

for target in [2, 4, 8]:

    path = os.path.join(
        RESULTS_DIR,
        f"results_dp{target}.json"
    )

    with open(path, "r") as f:
        saved = json.load(f)

    sigma = float(saved["noise_multiplier"])
    recorded_epsilon = float(saved["final_epsilon"])

    results = {
        "target_epsilon": target,
        "noise_multiplier": sigma,
        "recorded_epsilon": recorded_epsilon
    }

    for name, accountant_class in [
        ("rdp", RDPAccountant),
        ("prv", PRVAccountant)
    ]:
        try:
            accountant = accountant_class()

            # One homogeneous sampling/noise schedule.
            # Repeating step() allows Opacus to compress
            # consecutive identical accountant events.
            for _ in range(STEPS):
                accountant.step(
                    noise_multiplier=sigma,
                    sample_rate=SAMPLE_RATE
                )

            calculated = float(
                accountant.get_epsilon(delta=DELTA)
            )

            results[f"{name}_epsilon"] = calculated
            results[f"{name}_difference"] = (
                calculated - recorded_epsilon
            )

        except Exception as exc:
            results[f"{name}_epsilon"] = None
            results[f"{name}_difference"] = None
            results[f"{name}_error"] = str(exc)

    records.append(results)

df = pd.DataFrame(records)

print("\nRECOMPUTED PRIVACY BUDGETS")
print("-" * 68)
print(df.to_string(index=False))

for name in ["rdp", "prv"]:
    differences = [
        abs(r[f"{name}_difference"])
        for r in records
        if r.get(f"{name}_difference") is not None
        and math.isfinite(r[f"{name}_difference"])
    ]

    if len(differences) == 3:
        print(
            f"\n{name.upper()} maximum absolute difference:",
            max(differences)
        )

output_path = os.path.join(
    RESULTS_DIR,
    "dnabert_dpsat_accountant_verification.json"
)

with open(output_path, "w") as f:
    json.dump({
        "opacus_version": opacus.__version__,
        "sample_rate": SAMPLE_RATE,
        "steps": STEPS,
        "delta": DELTA,
        "results": records,
        "limitations": [
            "Recomputes epsilon assuming homogeneous Poisson sampling.",
            "Does not establish which accountant original training used.",
            "Does not audit original sampling or noise implementation.",
            "Does not verify the future DP-SAT training loop."
        ]
    }, f, indent=2)

print("\n" + "=" * 68)
print("CELL 48 COMPLETE")
print("=" * 68)
print("Saved:", output_path)

**Original source cell 69** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 51 - ACTUAL POISSON SAMPLING IMPLEMENTATION
# DP-SAT preparation
# ============================================================

import numpy as np
import torch
from torch.utils.data import Sampler, DataLoader

SEED = 42
EXPECTED_N = 42620
EXPECTED_BATCH_SIZE = 8
SAMPLE_RATE = EXPECTED_BATCH_SIZE / EXPECTED_N
TEST_STEPS = 1000

print("=" * 68)
print("CELL 51 - POISSON SAMPLING IMPLEMENTATION")
print("=" * 68)

assert "promoter_train_dataset" in globals(), (
    "Run Cell 50 first."
)

assert len(promoter_train_dataset) == EXPECTED_N

# ------------------------------------------------------------
# 1. Define independent Poisson batch sampler
# ------------------------------------------------------------

class PoissonBatchSampler(Sampler):
    """
    Each example is independently included in every step
    with probability sample_rate.

    Sampling is with replacement across steps.
    A batch may contain zero examples.
    """

    def __init__(
        self,
        dataset_size,
        sample_rate,
        num_steps,
        seed=42
    ):
        self.dataset_size = dataset_size
        self.sample_rate = sample_rate
        self.num_steps = num_steps
        self.seed = seed

        assert dataset_size > 0
        assert 0 < sample_rate <= 1
        assert num_steps > 0

    def __iter__(self):

        rng = np.random.default_rng(self.seed)

        for _ in range(self.num_steps):

            # Draw a binomial batch size, then sample
            # that many unique indices uniformly.
            #
            # This is distributionally equivalent to
            # independent Bernoulli inclusion for each
            # training record.

            batch_size = rng.binomial(
                self.dataset_size,
                self.sample_rate
            )

            if batch_size == 0:
                yield []
                continue

            indices = rng.choice(
                self.dataset_size,
                size=batch_size,
                replace=False
            )

            yield indices.tolist()

    def __len__(self):
        return self.num_steps


# ------------------------------------------------------------
# 2. Test the sampler
# ------------------------------------------------------------

sampler = PoissonBatchSampler(
    dataset_size=EXPECTED_N,
    sample_rate=SAMPLE_RATE,
    num_steps=TEST_STEPS,
    seed=SEED
)

batch_sizes = []
duplicate_batches = 0
invalid_indices = 0

for batch_indices in sampler:

    batch_sizes.append(len(batch_indices))

    if len(batch_indices) != len(set(batch_indices)):
        duplicate_batches += 1

    if any(
        idx < 0 or idx >= EXPECTED_N
        for idx in batch_indices
    ):
        invalid_indices += 1

batch_sizes = np.asarray(batch_sizes)

print("\n1. SAMPLING DISTRIBUTION")
print("-" * 68)

print("Training examples:", EXPECTED_N)
print("Sampling probability:", SAMPLE_RATE)
print("Expected batch size:", EXPECTED_BATCH_SIZE)
print("Observed mean batch size:", batch_sizes.mean())
print("Observed batch-size std:", batch_sizes.std())
print("Minimum batch size:", batch_sizes.min())
print("Maximum batch size:", batch_sizes.max())
print("Empty batches:", np.sum(batch_sizes == 0))

print("\n2. SAMPLER VALIDATION")
print("-" * 68)

print("Duplicate indices within a batch:", duplicate_batches)
print("Invalid indices:", invalid_indices)

assert duplicate_batches == 0
assert invalid_indices == 0

assert abs(
    batch_sizes.mean() - EXPECTED_BATCH_SIZE
) < 1.0, "Observed mean is unexpectedly far from 8."

# ------------------------------------------------------------
# 3. Create the actual DP-SAT training sampler
# ------------------------------------------------------------

FULL_STEPS = 15984

dpsat_sampler = PoissonBatchSampler(
    dataset_size=len(promoter_train_dataset),
    sample_rate=SAMPLE_RATE,
    num_steps=FULL_STEPS,
    seed=SEED
)

print("\n3. DP-SAT TRAINING SAMPLER")
print("-" * 68)

print("Full training steps:", len(dpsat_sampler))
print("Sampling probability:", dpsat_sampler.sample_rate)

# IMPORTANT:
# PyTorch's default DataLoader collator cannot handle
# empty batches. The real DP-SAT training loop must
# explicitly handle [] without skipping the privacy step.

print("\n" + "=" * 68)
print("CELL 51 COMPLETE - POISSON SAMPLER VERIFIED")
print("=" * 68)

**Original source cell 73** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 55 - FULL DP-SAT TRAINING: EPSILON = 4
# DNABERT-2 + BiTFiT + DP-SAT
# Same optimizer and learning rate as original DP-SGD
# ============================================================

import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoModel
from opacus.accountants import RDPAccountant

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
TARGET_EPSILON = 4.0

MODEL_ID = "zhihan1996/DNABERT-2-117M"

SIGMA = 0.4736328125
MAX_GRAD_NORM = 1.0
RHO = 0.01

LEARNING_RATE = 2e-4
DP_BATCH_SIZE = 8

N_TRAIN = 42620
SAMPLE_RATE = DP_BATCH_SIZE / N_TRAIN

DP_EPOCHS = 3
STEPS_PER_EPOCH = 5328
TOTAL_STEPS = DP_EPOCHS * STEPS_PER_EPOCH

DELTA = 1e-5

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "DNABERT_DP_Project/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

PREFIX = "dnabert_dpsat4"

RESUME_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_resume.pt"
)
BEST_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_best.pt"
)
FINAL_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_final.pt"
)
RESULTS_PATH = os.path.join(
    RESULTS_DIR, f"results_dpsat4.json"
)
HISTORY_PATH = os.path.join(
    RESULTS_DIR, f"history_dpsat4.csv"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable a Colab GPU."
assert "DNABERTBiTFiTClassifier" in globals()
assert "promoter_train_dataset" in globals()
assert "promoter_val_loader" in globals()
assert len(promoter_train_dataset) == N_TRAIN

# Avoid accidentally overwriting a completed run.
if os.path.exists(FINAL_PATH):
    raise RuntimeError(
        "A completed DP-SAT final checkpoint already exists. "
        "Move or rename it before starting another run."
    )

print("=" * 68)
print("FULL DP-SAT TRAINING: EPSILON = 4")
print("=" * 68)
print("Device:", torch.cuda.get_device_name(0))
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Noise multiplier:", SIGMA)
print("Sharpness radius:", RHO)
print("Total steps:", TOTAL_STEPS)
print("Results directory:", RESULTS_DIR)

# ------------------------------------------------------------
# 2. Clear previous pilot model
# ------------------------------------------------------------

for name in [
    "dpsat_pilot_model",
    "dpsat_model",
    "dpsat_encoder",
    "optimizer"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Initialize model
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

dpsat_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dpsat_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        if hasattr(module, "dropout"):
            module.dropout.p = 0.1

dpsat_model = DNABERTBiTFiTClassifier(
    dpsat_encoder,
    num_labels=2
).to(device)

# Match original DP-SGD evaluation-mode training.
dpsat_model.eval()

trainable = [
    (name, p)
    for name, p in dpsat_model.named_parameters()
    if p.requires_grad
]

assert sum(p.numel() for _, p in trainable) == 66818
assert len(trainable) == 63

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=LEARNING_RATE
)

accountant = RDPAccountant()

# Same sampling generator and seed as original DP-SGD.
sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

# Previous privatized gradient used for DP-SAT perturbation.
previous_private_gradients = None

history = []
best_accuracy = -1.0
best_step = None
completed_steps = 0
start_epoch = 0
previous_elapsed = 0.0

# ------------------------------------------------------------
# 4. Resume from an epoch checkpoint, if available
# ------------------------------------------------------------

if os.path.isfile(RESUME_PATH):

    print("\nFound existing DP-SAT resume checkpoint.")

    checkpoint = torch.load(
        RESUME_PATH,
        map_location="cpu",
        weights_only=False
    )

    assert checkpoint["configuration"]["sigma"] == SIGMA
    assert checkpoint["configuration"]["rho"] == RHO
    assert checkpoint["configuration"]["learning_rate"] == LEARNING_RATE
    assert checkpoint["configuration"]["sample_rate"] == SAMPLE_RATE
    assert checkpoint["configuration"]["steps_per_epoch"] == STEPS_PER_EPOCH

    dpsat_model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    # Move optimizer state tensors to the current GPU.
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

    accountant.load_state_dict(
        checkpoint["accountant_state"]
    )

    sampling_generator.set_state(
        checkpoint["sampling_rng_state"]
    )

    torch.set_rng_state(checkpoint["torch_rng_state"])

    if checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    previous_private_gradients = (
        checkpoint["previous_private_gradients"]
    )

    if previous_private_gradients is not None:
        previous_private_gradients = [
            g.to(device)
            for g in previous_private_gradients
        ]

    history = checkpoint["history"]
    best_accuracy = checkpoint["best_accuracy"]
    best_step = checkpoint["best_step"]
    completed_steps = checkpoint["completed_steps"]
    start_epoch = checkpoint["completed_epochs"]
    previous_elapsed = checkpoint["elapsed_seconds"]

    print("Resuming from epoch:", start_epoch + 1)
    print("Completed steps:", completed_steps)

else:
    print("\nStarting a new DP-SAT run.")

# ------------------------------------------------------------
# 5. Helper: save resumable checkpoint
# ------------------------------------------------------------

def save_dpsat_resume(
    completed_epochs,
    elapsed_seconds
):
    checkpoint = {
        "model_state": dpsat_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "accountant_state": accountant.state_dict(),
        "sampling_rng_state": sampling_generator.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
        "previous_private_gradients": (
            None if previous_private_gradients is None
            else [
                g.detach().cpu()
                for g in previous_private_gradients
            ]
        ),
        "completed_epochs": completed_epochs,
        "completed_steps": completed_steps,
        "best_accuracy": best_accuracy,
        "best_step": best_step,
        "history": history,
        "elapsed_seconds": elapsed_seconds,
        "configuration": {
            "sigma": SIGMA,
            "rho": RHO,
            "learning_rate": LEARNING_RATE,
            "sample_rate": SAMPLE_RATE,
            "steps_per_epoch": STEPS_PER_EPOCH
        }
    }

    temporary_path = RESUME_PATH + ".tmp"

    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, RESUME_PATH)

# ------------------------------------------------------------
# 6. Training
# ------------------------------------------------------------

session_start = time.time()

for epoch in range(start_epoch, DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0
    epoch_empty_batches = 0

    dpsat_model.eval()

    for step in range(STEPS_PER_EPOCH):

        # A. Independent Poisson sampling.
        # Identical sampling procedure to original DP-SGD.
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        if len(selected) == 0:
            epoch_empty_batches += 1

        # B. DP-SAT perturbation from PREVIOUS private gradient.
        perturbations = []

        if previous_private_gradients is not None:

            with torch.no_grad():

                previous_norm = torch.linalg.vector_norm(
                    torch.stack([
                        torch.linalg.vector_norm(g)
                        for g in previous_private_gradients
                    ])
                )

                scale = RHO / (
                    previous_norm.item() + 1e-12
                )

                for (_, p), g in zip(
                    trainable,
                    previous_private_gradients
                ):
                    delta = g * scale
                    p.add_(delta)
                    perturbations.append(delta)

        # C. Compute clipped per-example gradients at
        # the perturbed parameters.
        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = promoter_train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dpsat_model.zero_grad(set_to_none=True)

            logits = dpsat_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(
                        p.grad.detach()
                    )
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (
                    grad_norm.item() + 1e-12
                )
            )

            if factor < 1.0:
                epoch_clipped += 1

            with torch.no_grad():
                for j, (_, p) in enumerate(trainable):
                    clipped_sums[j].add_(
                        p.grad.detach(),
                        alpha=factor
                    )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        # D. Restore original unperturbed parameters.
        with torch.no_grad():
            for (_, p), delta in zip(
                trainable,
                perturbations
            ):
                p.sub_(delta)

        # E. Privatize the gradient ONCE.
        optimizer.zero_grad(set_to_none=True)

        current_private_gradients = []

        with torch.no_grad():

            for j, (_, p) in enumerate(trainable):

                noise = torch.randn_like(p) * (
                    SIGMA * MAX_GRAD_NORM
                )

                private_grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

                p.grad = private_grad

                current_private_gradients.append(
                    private_grad.detach().clone()
                )

        # F. Apply AdamW update to original parameters.
        optimizer.step()

        # G. Store current private gradient for next step.
        previous_private_gradients = (
            current_private_gradients
        )

        # H. Account for every step, including empty batches.
        accountant.step(
            noise_multiplier=SIGMA,
            sample_rate=SAMPLE_RATE
        )

        completed_steps += 1

        # Occasional progress reporting.
        if (step + 1) % 1000 == 0:

            elapsed = (
                previous_elapsed
                + time.time()
                - session_start
            )

            print(
                f"Epoch {epoch+1}/{DP_EPOCHS} | "
                f"Step {step+1}/{STEPS_PER_EPOCH} | "
                f"Total {completed_steps}/{TOTAL_STEPS} | "
                f"Elapsed {elapsed/60:.1f} min",
                flush=True
            )

    # --------------------------------------------------------
    # 7. Validation after each epoch
    # --------------------------------------------------------

    dpsat_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_val in promoter_val_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dpsat_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            ).item()

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(
        delta=DELTA
    )

    elapsed = (
        previous_elapsed
        + time.time()
        - session_start
    )

    history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "empty_batches": epoch_empty_batches,
        "elapsed_seconds": elapsed
    })

    # Save best model.
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_step = completed_steps

        torch.save(
            dpsat_model.state_dict(),
            BEST_PATH
        )

    # Save resumable training state after every epoch.
    save_dpsat_resume(
        completed_epochs=epoch + 1,
        elapsed_seconds=elapsed
    )

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    print(
        f"\nEpoch {epoch+1}/{DP_EPOCHS} COMPLETE | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min\n",
        flush=True
    )

# ------------------------------------------------------------
# 8. Save final results
# ------------------------------------------------------------

torch.save(
    dpsat_model.state_dict(),
    FINAL_PATH
)

final_epsilon = accountant.get_epsilon(
    delta=DELTA
)

total_elapsed = (
    previous_elapsed
    + time.time()
    - session_start
)

results_dpsat4 = {
    "method": "DP-SAT",
    "target_epsilon": TARGET_EPSILON,
    "noise_multiplier": SIGMA,
    "final_epsilon": float(final_epsilon),
    "delta": DELTA,
    "rho": RHO,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "expected_batch_size": DP_BATCH_SIZE,
    "sample_rate": SAMPLE_RATE,
    "clip_norm": MAX_GRAD_NORM,
    "total_steps": completed_steps,
    "best_validation_accuracy": float(best_accuracy),
    "best_checkpoint_step": int(best_step),
    "training_history": history,
    "total_runtime_minutes": total_elapsed / 60,
    "privacy_accountant": "Opacus RDP",
    "noise_rng": "PyTorch default RNG"
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results_dpsat4, f, indent=2)

pd.DataFrame(history).to_csv(
    HISTORY_PATH,
    index=False
)

print("\n" + "=" * 68)
print("DP-SAT EPSILON = 4 COMPLETE")
print("=" * 68)
print("Best validation accuracy:", f"{best_accuracy:.2%}")
print("Final epsilon:", f"{final_epsilon:.4f}")
print("Total runtime:", f"{total_elapsed/60:.1f} minutes")
print("Best checkpoint:", BEST_PATH)
print("Final checkpoint:", FINAL_PATH)
print("Results:", RESULTS_PATH)

**Original source cell 79** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 57 - FULL DP-SAT TRAINING: EPSILON = 2
# DNABERT-2 + BiTFiT + DP-SAT
# Same optimizer and learning rate as original DP-SGD
# ============================================================

import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoModel
from opacus.accountants import RDPAccountant

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
TARGET_EPSILON = 2.0

MODEL_ID = "zhihan1996/DNABERT-2-117M"

SIGMA = 0.592041015625
MAX_GRAD_NORM = 1.0
RHO = 0.01

LEARNING_RATE = 2e-4
DP_BATCH_SIZE = 8

N_TRAIN = 42620
SAMPLE_RATE = DP_BATCH_SIZE / N_TRAIN

DP_EPOCHS = 3
STEPS_PER_EPOCH = 5328
TOTAL_STEPS = DP_EPOCHS * STEPS_PER_EPOCH

DELTA = 1e-5

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "DNABERT_DP_Project/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

PREFIX = "dnabert_dpsat2"

RESUME_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_resume.pt"
)
BEST_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_best.pt"
)
FINAL_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_final.pt"
)
RESULTS_PATH = os.path.join(
    RESULTS_DIR, f"results_dpsat2.json"
)
HISTORY_PATH = os.path.join(
    RESULTS_DIR, f"history_dpsat2.csv"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable a Colab GPU."
assert "DNABERTBiTFiTClassifier" in globals()
assert "promoter_train_dataset" in globals()
assert "promoter_val_loader" in globals()
assert len(promoter_train_dataset) == N_TRAIN

# Avoid accidentally overwriting a completed run.
if os.path.exists(FINAL_PATH):
    raise RuntimeError(
        "A completed DP-SAT final checkpoint already exists. "
        "Move or rename it before starting another run."
    )

print("=" * 68)
print("FULL DP-SAT TRAINING: EPSILON = 2")
print("=" * 68)
print("Device:", torch.cuda.get_device_name(0))
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Noise multiplier:", SIGMA)
print("Sharpness radius:", RHO)
print("Total steps:", TOTAL_STEPS)
print("Results directory:", RESULTS_DIR)

# ------------------------------------------------------------
# 2. Clear previous pilot model
# ------------------------------------------------------------

for name in [
    "dpsat_pilot_model",
    "dpsat_model",
    "dpsat_encoder",
    "optimizer"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Initialize model
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

dpsat_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dpsat_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        if hasattr(module, "dropout"):
            module.dropout.p = 0.1

dpsat_model = DNABERTBiTFiTClassifier(
    dpsat_encoder,
    num_labels=2
).to(device)

# Match original DP-SGD evaluation-mode training.
dpsat_model.eval()

trainable = [
    (name, p)
    for name, p in dpsat_model.named_parameters()
    if p.requires_grad
]

assert sum(p.numel() for _, p in trainable) == 66818
assert len(trainable) == 63

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=LEARNING_RATE
)

accountant = RDPAccountant()

# Same sampling generator and seed as original DP-SGD.
sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

# Previous privatized gradient used for DP-SAT perturbation.
previous_private_gradients = None

history = []
best_accuracy = -1.0
best_step = None
completed_steps = 0
start_epoch = 0
previous_elapsed = 0.0

# ------------------------------------------------------------
# 4. Resume from an epoch checkpoint, if available
# ------------------------------------------------------------

if os.path.isfile(RESUME_PATH):

    print("\nFound existing DP-SAT resume checkpoint.")

    checkpoint = torch.load(
        RESUME_PATH,
        map_location="cpu",
        weights_only=False
    )

    assert checkpoint["configuration"]["sigma"] == SIGMA
    assert checkpoint["configuration"]["rho"] == RHO
    assert checkpoint["configuration"]["learning_rate"] == LEARNING_RATE
    assert checkpoint["configuration"]["sample_rate"] == SAMPLE_RATE
    assert checkpoint["configuration"]["steps_per_epoch"] == STEPS_PER_EPOCH

    dpsat_model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    # Move optimizer state tensors to the current GPU.
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

    accountant.load_state_dict(
        checkpoint["accountant_state"]
    )

    sampling_generator.set_state(
        checkpoint["sampling_rng_state"]
    )

    torch.set_rng_state(checkpoint["torch_rng_state"])

    if checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    previous_private_gradients = (
        checkpoint["previous_private_gradients"]
    )

    if previous_private_gradients is not None:
        previous_private_gradients = [
            g.to(device)
            for g in previous_private_gradients
        ]

    history = checkpoint["history"]
    best_accuracy = checkpoint["best_accuracy"]
    best_step = checkpoint["best_step"]
    completed_steps = checkpoint["completed_steps"]
    start_epoch = checkpoint["completed_epochs"]
    previous_elapsed = checkpoint["elapsed_seconds"]

    print("Resuming from epoch:", start_epoch + 1)
    print("Completed steps:", completed_steps)

else:
    print("\nStarting a new DP-SAT run.")

# ------------------------------------------------------------
# 5. Helper: save resumable checkpoint
# ------------------------------------------------------------

def save_dpsat_resume(
    completed_epochs,
    elapsed_seconds
):
    checkpoint = {
        "model_state": dpsat_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "accountant_state": accountant.state_dict(),
        "sampling_rng_state": sampling_generator.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
        "previous_private_gradients": (
            None if previous_private_gradients is None
            else [
                g.detach().cpu()
                for g in previous_private_gradients
            ]
        ),
        "completed_epochs": completed_epochs,
        "completed_steps": completed_steps,
        "best_accuracy": best_accuracy,
        "best_step": best_step,
        "history": history,
        "elapsed_seconds": elapsed_seconds,
        "configuration": {
            "sigma": SIGMA,
            "rho": RHO,
            "learning_rate": LEARNING_RATE,
            "sample_rate": SAMPLE_RATE,
            "steps_per_epoch": STEPS_PER_EPOCH
        }
    }

    temporary_path = RESUME_PATH + ".tmp"

    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, RESUME_PATH)

# ------------------------------------------------------------
# 6. Training
# ------------------------------------------------------------

session_start = time.time()

for epoch in range(start_epoch, DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0
    epoch_empty_batches = 0

    dpsat_model.eval()

    for step in range(STEPS_PER_EPOCH):

        # A. Independent Poisson sampling.
        # Identical sampling procedure to original DP-SGD.
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        if len(selected) == 0:
            epoch_empty_batches += 1

        # B. DP-SAT perturbation from PREVIOUS private gradient.
        perturbations = []

        if previous_private_gradients is not None:

            with torch.no_grad():

                previous_norm = torch.linalg.vector_norm(
                    torch.stack([
                        torch.linalg.vector_norm(g)
                        for g in previous_private_gradients
                    ])
                )

                scale = RHO / (
                    previous_norm.item() + 1e-12
                )

                for (_, p), g in zip(
                    trainable,
                    previous_private_gradients
                ):
                    delta = g * scale
                    p.add_(delta)
                    perturbations.append(delta)

        # C. Compute clipped per-example gradients at
        # the perturbed parameters.
        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = promoter_train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dpsat_model.zero_grad(set_to_none=True)

            logits = dpsat_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(
                        p.grad.detach()
                    )
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (
                    grad_norm.item() + 1e-12
                )
            )

            if factor < 1.0:
                epoch_clipped += 1

            with torch.no_grad():
                for j, (_, p) in enumerate(trainable):
                    clipped_sums[j].add_(
                        p.grad.detach(),
                        alpha=factor
                    )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        # D. Restore original unperturbed parameters.
        with torch.no_grad():
            for (_, p), delta in zip(
                trainable,
                perturbations
            ):
                p.sub_(delta)

        # E. Privatize the gradient ONCE.
        optimizer.zero_grad(set_to_none=True)

        current_private_gradients = []

        with torch.no_grad():

            for j, (_, p) in enumerate(trainable):

                noise = torch.randn_like(p) * (
                    SIGMA * MAX_GRAD_NORM
                )

                private_grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

                p.grad = private_grad

                current_private_gradients.append(
                    private_grad.detach().clone()
                )

        # F. Apply AdamW update to original parameters.
        optimizer.step()

        # G. Store current private gradient for next step.
        previous_private_gradients = (
            current_private_gradients
        )

        # H. Account for every step, including empty batches.
        accountant.step(
            noise_multiplier=SIGMA,
            sample_rate=SAMPLE_RATE
        )

        completed_steps += 1

        # Occasional progress reporting.
        if (step + 1) % 1000 == 0:

            elapsed = (
                previous_elapsed
                + time.time()
                - session_start
            )

            print(
                f"Epoch {epoch+1}/{DP_EPOCHS} | "
                f"Step {step+1}/{STEPS_PER_EPOCH} | "
                f"Total {completed_steps}/{TOTAL_STEPS} | "
                f"Elapsed {elapsed/60:.1f} min",
                flush=True
            )

    # --------------------------------------------------------
    # 7. Validation after each epoch
    # --------------------------------------------------------

    dpsat_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_val in promoter_val_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dpsat_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            ).item()

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(
        delta=DELTA
    )

    elapsed = (
        previous_elapsed
        + time.time()
        - session_start
    )

    history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "empty_batches": epoch_empty_batches,
        "elapsed_seconds": elapsed
    })

    # Save best model.
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_step = completed_steps

        torch.save(
            dpsat_model.state_dict(),
            BEST_PATH
        )

    # Save resumable training state after every epoch.
    save_dpsat_resume(
        completed_epochs=epoch + 1,
        elapsed_seconds=elapsed
    )

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    print(
        f"\nEpoch {epoch+1}/{DP_EPOCHS} COMPLETE | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min\n",
        flush=True
    )

# ------------------------------------------------------------
# 8. Save final results
# ------------------------------------------------------------

torch.save(
    dpsat_model.state_dict(),
    FINAL_PATH
)

final_epsilon = accountant.get_epsilon(
    delta=DELTA
)

total_elapsed = (
    previous_elapsed
    + time.time()
    - session_start
)

results_dpsat2 = {
    "method": "DP-SAT",
    "target_epsilon": TARGET_EPSILON,
    "noise_multiplier": SIGMA,
    "final_epsilon": float(final_epsilon),
    "delta": DELTA,
    "rho": RHO,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "expected_batch_size": DP_BATCH_SIZE,
    "sample_rate": SAMPLE_RATE,
    "clip_norm": MAX_GRAD_NORM,
    "total_steps": completed_steps,
    "best_validation_accuracy": float(best_accuracy),
    "best_checkpoint_step": int(best_step),
    "training_history": history,
    "total_runtime_minutes": total_elapsed / 60,
    "privacy_accountant": "Opacus RDP",
    "noise_rng": "PyTorch default RNG"
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results_dpsat2, f, indent=2)

pd.DataFrame(history).to_csv(
    HISTORY_PATH,
    index=False
)

print("\n" + "=" * 68)
print("DP-SAT EPSILON = 2 COMPLETE")
print("=" * 68)
print("Best validation accuracy:", f"{best_accuracy:.2%}")
print("Final epsilon:", f"{final_epsilon:.4f}")
print("Total runtime:", f"{total_elapsed/60:.1f} minutes")
print("Best checkpoint:", BEST_PATH)
print("Final checkpoint:", FINAL_PATH)
print("Results:", RESULTS_PATH)

**Original source cell 82** — retained with its original logic.


In [ ]:
# ============================================================
# CELL 59 - FULL DP-SAT TRAINING: EPSILON = 8
# DNABERT-2 + BiTFiT + DP-SAT
# Same optimizer and learning rate as original DP-SGD
# ============================================================

import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import AutoModel
from opacus.accountants import RDPAccountant

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42
TARGET_EPSILON = 8.0

MODEL_ID = "zhihan1996/DNABERT-2-117M"

SIGMA = 0.389404296875
MAX_GRAD_NORM = 1.0
RHO = 0.01

LEARNING_RATE = 2e-4
DP_BATCH_SIZE = 8

N_TRAIN = 42620
SAMPLE_RATE = DP_BATCH_SIZE / N_TRAIN

DP_EPOCHS = 3
STEPS_PER_EPOCH = 5328
TOTAL_STEPS = DP_EPOCHS * STEPS_PER_EPOCH

DELTA = 1e-5

RESULTS_DIR = (
    "/content/drive/MyDrive/"
    "DNABERT_DP_Project/results"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

PREFIX = "dnabert_dpsat8"

RESUME_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_resume.pt"
)

BEST_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_best.pt"
)

FINAL_PATH = os.path.join(
    RESULTS_DIR, f"{PREFIX}_final.pt"
)

RESULTS_PATH = os.path.join(
    RESULTS_DIR, "results_dpsat8.json"
)

HISTORY_PATH = os.path.join(
    RESULTS_DIR, "history_dpsat8.csv"
)

# Safety checks
assert TARGET_EPSILON == 8.0
assert SIGMA == 0.389404296875
assert PREFIX == "dnabert_dpsat8"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", "Enable a Colab GPU."
assert "DNABERTBiTFiTClassifier" in globals()
assert "promoter_train_dataset" in globals()
assert "promoter_val_loader" in globals()
assert len(promoter_train_dataset) == N_TRAIN

if os.path.exists(FINAL_PATH):
    raise RuntimeError(
        "A completed DP-SAT epsilon=8 checkpoint exists. "
        "Do not overwrite a completed experiment."
    )

print("=" * 68)
print("FULL DP-SAT TRAINING: EPSILON = 8")
print("=" * 68)
print("Device:", torch.cuda.get_device_name(0))
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Noise multiplier:", SIGMA)
print("Sharpness radius:", RHO)
print("Total steps:", TOTAL_STEPS)
print("Results directory:", RESULTS_DIR)

# ------------------------------------------------------------
# 2. Clear previous training models
# ------------------------------------------------------------

for name in [
    "dpsat_pilot_model",
    "dpsat_model",
    "dpsat_encoder",
    "optimizer"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Initialize model
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

dpsat_encoder = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    add_pooling_layer=False
)

for module in dpsat_encoder.modules():
    if module.__class__.__name__ == "BertUnpadSelfAttention":
        module.p_dropout = 0.1
        if hasattr(module, "dropout"):
            module.dropout.p = 0.1

dpsat_model = DNABERTBiTFiTClassifier(
    dpsat_encoder,
    num_labels=2
).to(device)

# Match original DP-SGD evaluation-mode training.
dpsat_model.eval()

trainable = [
    (name, p)
    for name, p in dpsat_model.named_parameters()
    if p.requires_grad
]

assert sum(p.numel() for _, p in trainable) == 66818
assert len(trainable) == 63

optimizer = torch.optim.AdamW(
    [p for _, p in trainable],
    lr=LEARNING_RATE
)

accountant = RDPAccountant()

sampling_generator = torch.Generator(device="cpu")
sampling_generator.manual_seed(12345)

previous_private_gradients = None

history = []
best_accuracy = -1.0
best_step = None
completed_steps = 0
start_epoch = 0
previous_elapsed = 0.0

# ------------------------------------------------------------
# 4. Resume from checkpoint if available
# ------------------------------------------------------------

if os.path.isfile(RESUME_PATH):

    print("\nFound existing DP-SAT epsilon=8 checkpoint.")

    checkpoint = torch.load(
        RESUME_PATH,
        map_location="cpu",
        weights_only=False
    )

    assert checkpoint["configuration"]["sigma"] == SIGMA
    assert checkpoint["configuration"]["rho"] == RHO
    assert checkpoint["configuration"]["learning_rate"] == LEARNING_RATE
    assert checkpoint["configuration"]["sample_rate"] == SAMPLE_RATE
    assert checkpoint["configuration"]["steps_per_epoch"] == STEPS_PER_EPOCH

    dpsat_model.load_state_dict(
        checkpoint["model_state"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state"]
    )

    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)

    accountant.load_state_dict(
        checkpoint["accountant_state"]
    )

    sampling_generator.set_state(
        checkpoint["sampling_rng_state"]
    )

    torch.set_rng_state(
        checkpoint["torch_rng_state"]
    )

    if checkpoint.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    previous_private_gradients = checkpoint[
        "previous_private_gradients"
    ]

    if previous_private_gradients is not None:
        previous_private_gradients = [
            g.to(device)
            for g in previous_private_gradients
        ]

    history = checkpoint["history"]
    best_accuracy = checkpoint["best_accuracy"]
    best_step = checkpoint["best_step"]
    completed_steps = checkpoint["completed_steps"]
    start_epoch = checkpoint["completed_epochs"]
    previous_elapsed = checkpoint["elapsed_seconds"]

    print("Resuming from epoch:", start_epoch + 1)
    print("Completed steps:", completed_steps)

else:
    print("\nStarting a new DP-SAT epsilon=8 run.")

# ------------------------------------------------------------
# 5. Helper: save resumable checkpoint
# ------------------------------------------------------------

def save_dpsat_resume(completed_epochs, elapsed_seconds):

    checkpoint = {
        "model_state": dpsat_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "accountant_state": accountant.state_dict(),
        "sampling_rng_state": sampling_generator.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state_all(),
        "previous_private_gradients": (
            None if previous_private_gradients is None
            else [
                g.detach().cpu()
                for g in previous_private_gradients
            ]
        ),
        "completed_epochs": completed_epochs,
        "completed_steps": completed_steps,
        "best_accuracy": best_accuracy,
        "best_step": best_step,
        "history": history,
        "elapsed_seconds": elapsed_seconds,
        "configuration": {
            "sigma": SIGMA,
            "rho": RHO,
            "learning_rate": LEARNING_RATE,
            "sample_rate": SAMPLE_RATE,
            "steps_per_epoch": STEPS_PER_EPOCH
        }
    }

    temporary_path = RESUME_PATH + ".tmp"

    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, RESUME_PATH)

# ------------------------------------------------------------
# 6. Full DP-SAT training
# ------------------------------------------------------------

session_start = time.time()

for epoch in range(start_epoch, DP_EPOCHS):

    epoch_losses = []
    epoch_clipped = 0
    epoch_samples = 0
    epoch_empty_batches = 0

    dpsat_model.eval()

    for step in range(STEPS_PER_EPOCH):

        # A. Independent Poisson sampling
        selected = torch.nonzero(
            torch.rand(
                N_TRAIN,
                generator=sampling_generator
            ) < SAMPLE_RATE,
            as_tuple=True
        )[0].tolist()

        if len(selected) == 0:
            epoch_empty_batches += 1

        # B. DP-SAT perturbation using previous private gradient
        perturbations = []

        if previous_private_gradients is not None:

            with torch.no_grad():

                previous_norm = torch.linalg.vector_norm(
                    torch.stack([
                        torch.linalg.vector_norm(g)
                        for g in previous_private_gradients
                    ])
                )

                scale = RHO / (
                    previous_norm.item() + 1e-12
                )

                for (_, p), g in zip(
                    trainable,
                    previous_private_gradients
                ):
                    delta = g * scale
                    p.add_(delta)
                    perturbations.append(delta)

        # C. Compute clipped per-example gradients
        clipped_sums = [
            torch.zeros_like(p)
            for _, p in trainable
        ]

        for idx in selected:

            example = promoter_train_dataset[idx]

            input_ids = torch.tensor(
                [example["input_ids"]],
                dtype=torch.long,
                device=device
            )

            attention_mask = torch.tensor(
                [example["attention_mask"]],
                dtype=torch.long,
                device=device
            )

            labels = torch.tensor(
                [example["labels"]],
                dtype=torch.long,
                device=device
            )

            dpsat_model.zero_grad(set_to_none=True)

            logits = dpsat_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = F.cross_entropy(logits, labels)
            loss.backward()

            grad_norm = torch.linalg.vector_norm(
                torch.stack([
                    torch.linalg.vector_norm(
                        p.grad.detach()
                    )
                    for _, p in trainable
                ])
            )

            factor = min(
                1.0,
                MAX_GRAD_NORM / (
                    grad_norm.item() + 1e-12
                )
            )

            if factor < 1.0:
                epoch_clipped += 1

            with torch.no_grad():
                for j, (_, p) in enumerate(trainable):
                    clipped_sums[j].add_(
                        p.grad.detach(),
                        alpha=factor
                    )

            epoch_losses.append(loss.item())
            epoch_samples += 1

        # D. Restore unperturbed parameters
        with torch.no_grad():
            for (_, p), delta in zip(
                trainable,
                perturbations
            ):
                p.sub_(delta)

        # E. Add Gaussian noise once
        optimizer.zero_grad(set_to_none=True)

        current_private_gradients = []

        with torch.no_grad():

            for j, (_, p) in enumerate(trainable):

                noise = torch.randn_like(p) * (
                    SIGMA * MAX_GRAD_NORM
                )

                private_grad = (
                    clipped_sums[j] + noise
                ) / DP_BATCH_SIZE

                p.grad = private_grad

                current_private_gradients.append(
                    private_grad.detach().clone()
                )

        # F. Update original parameters
        optimizer.step()

        # G. Store privatized gradient
        previous_private_gradients = (
            current_private_gradients
        )

        # H. Privacy accounting
        accountant.step(
            noise_multiplier=SIGMA,
            sample_rate=SAMPLE_RATE
        )

        completed_steps += 1

        if (step + 1) % 1000 == 0:

            elapsed = (
                previous_elapsed
                + time.time()
                - session_start
            )

            print(
                f"Epoch {epoch+1}/{DP_EPOCHS} | "
                f"Step {step+1}/{STEPS_PER_EPOCH} | "
                f"Total {completed_steps}/{TOTAL_STEPS} | "
                f"Elapsed {elapsed/60:.1f} min",
                flush=True
            )

    # --------------------------------------------------------
    # 7. Validation after each epoch
    # --------------------------------------------------------

    dpsat_model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_val in promoter_val_loader:

            ids = batch_val["input_ids"].to(device)
            mask = batch_val["attention_mask"].to(device)
            labels = batch_val["labels"].to(device).long()

            logits = dpsat_model(
                input_ids=ids,
                attention_mask=mask
            )

            val_loss += F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            ).item()

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss /= val_total

    current_epsilon = accountant.get_epsilon(
        delta=DELTA
    )

    elapsed = (
        previous_elapsed
        + time.time()
        - session_start
    )

    history.append({
        "epoch": epoch + 1,
        "validation_accuracy": val_accuracy,
        "validation_loss": val_loss,
        "epsilon": current_epsilon,
        "samples_processed": epoch_samples,
        "clipped": epoch_clipped,
        "empty_batches": epoch_empty_batches,
        "elapsed_seconds": elapsed
    })

    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy
        best_step = completed_steps

        torch.save(
            dpsat_model.state_dict(),
            BEST_PATH
        )

    save_dpsat_resume(
        completed_epochs=epoch + 1,
        elapsed_seconds=elapsed
    )

    pd.DataFrame(history).to_csv(
        HISTORY_PATH,
        index=False
    )

    print(
        f"\nEpoch {epoch+1}/{DP_EPOCHS} COMPLETE | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f} | "
        f"Epsilon: {current_epsilon:.4f} | "
        f"Time: {elapsed/60:.1f} min\n",
        flush=True
    )

# ------------------------------------------------------------
# 8. Save final results
# ------------------------------------------------------------

torch.save(
    dpsat_model.state_dict(),
    FINAL_PATH
)

final_epsilon = accountant.get_epsilon(
    delta=DELTA
)

total_elapsed = (
    previous_elapsed
    + time.time()
    - session_start
)

results_dpsat8 = {
    "method": "DP-SAT",
    "target_epsilon": TARGET_EPSILON,
    "noise_multiplier": SIGMA,
    "final_epsilon": float(final_epsilon),
    "delta": DELTA,
    "rho": RHO,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "expected_batch_size": DP_BATCH_SIZE,
    "sample_rate": SAMPLE_RATE,
    "clip_norm": MAX_GRAD_NORM,
    "total_steps": completed_steps,
    "best_validation_accuracy": float(best_accuracy),
    "best_checkpoint_step": int(best_step),
    "training_history": history,
    "total_runtime_minutes": total_elapsed / 60,
    "privacy_accountant": "Opacus RDP",
    "noise_rng": "PyTorch default RNG"
}

with open(RESULTS_PATH, "w") as f:
    json.dump(results_dpsat8, f, indent=2)

pd.DataFrame(history).to_csv(
    HISTORY_PATH,
    index=False
)

print("\n" + "=" * 68)
print("DP-SAT EPSILON = 8 COMPLETE")
print("=" * 68)
print("Best validation accuracy:", f"{best_accuracy:.2%}")
print("Final epsilon:", f"{final_epsilon:.4f}")
print("Total runtime:", f"{total_elapsed/60:.1f} minutes")
print("Best checkpoint:", BEST_PATH)
print("Final checkpoint:", FINAL_PATH)
print("Results:", RESULTS_PATH)


## Verification status
- Notebook JSON and code-cell Python syntax were checked (excluding Colab magics).
- **No GPU training or end-to-end runtime verification was performed.**
- Checkpoint references and privacy accounting are retained from original experiments, not independently certified.
- Avoid rerunning if a completed checkpoint exists.